# 🎬 IMDB Sentiment Analysis using RNN
Classify movie reviews as Positive or Negative using a Recurrent Neural Network (RNN) built with PyTorch.

## 1. Load Data

In [5]:
import pandas as pd

# load the dataset
df = pd.read_csv("IMDB Dataset.csv", on_bad_lines='skip', engine='python')

print(df.shape)       # total rows and columns
df.head()

(41223, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
# check for missing values
df.isnull().sum()

,0
review,0
sentiment,0


In [7]:
# remove duplicate rows
df.drop_duplicates(inplace=True)
print(df.shape)

(40946, 2)


## 2. Text Preprocessing

### Step 1 — Convert to Lowercase

In [8]:
# convert all text to lowercase so 'Good' and 'good' are treated the same
df["review"] = df["review"].str.lower()

### Step 2 — Remove URLs

In [9]:
import re

# remove any URLs starting with http from the review text
def remove_urls(text):
    text = re.sub(r"http\S+", "", text)   # replace URL with empty string
    return text

df["review"] = df["review"].apply(remove_urls)

### Step 3 — Remove Punctuations

In [10]:
# keep only letters, digits and spaces — remove everything else (!, ?, . etc)
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### Step 4 — Remove HTML Tags

In [11]:
# IMDB reviews contain HTML tags like <br> — remove them
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)   # match anything between < and >
    return text

df["review"] = df["review"].apply(remove_html)

### Step 5 — Remove Stopwords

In [12]:
import nltk

# download required NLTK packages
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# stopwords are common words like 'the', 'is', 'a' that carry no sentiment meaning
def remove_stopwords(text):
    tokens = word_tokenize(text)               # split sentence into list of words
    stop_words = stopwords.words("english")    # list of ~179 common English words

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")      # remove stopword from text

    return text

df["review"] = df["review"].apply(remove_stopwords)
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### Step 6 — Stemming

In [14]:
# stemming reduces words to their root form
# running -> run, played -> play, movies -> movi
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)         # get root form of the word
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)             # join list back into a string

df["review"] = df["review"].apply(stemming)
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### Step 7 — Label Encoding

In [15]:
from sklearn.preprocessing import LabelEncoder

# convert text labels to numbers: 'negative' -> 0, 'positive' -> 1
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]   # target variable
print(y.value_counts())

sentiment
1    20524
0    20422
Name: count, dtype: int64


### Step 8 — TF-IDF Vectorization

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# convert text reviews into numerical feature vectors
# max_features=5000 -> keep only top 5000 most important words
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

print("X shape:", X.shape)   # (num_reviews, 5000)

X shape: (40946, 5000)


## 3. Train-Test Split & DataLoaders

In [17]:
from sklearn.model_selection import train_test_split

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (32756, 5000)
Test : (8190, 5000)


In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# TF-IDF returns sparse matrix — convert to dense array for PyTorch
X_train = X_train.toarray()
X_test = X_test.toarray()

In [19]:
# wrap X and y together into TensorDataset so DataLoader can use them
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),          # features as float tensor
    torch.from_numpy(y_train.values).float()    # labels as float tensor (needed for BCELoss)
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [20]:
# feed data in batches of 64 samples at a time
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## 4. Build the RNN Model

In [21]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer — processes the input sequence
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer — maps hidden state to output (1 value)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # initialize hidden state with zeros at the start
        # shape: (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        # pass input through RNN
        # out: hidden states for all time steps (batch, seq_len, hidden_size)
        # _  : final hidden state (not needed here)
        out, _ = self.rnn(x, h0)

        # take only the last time step's output — model has seen full input by then
        out = self.fc(out[:, -1, :])
        return out

In [22]:
input_size = X_train.shape[1]   # 5000 (TF-IDF features)

model = RNN(input_size)

# BCELoss for binary classification (positive/negative)
criterion = nn.BCELoss()

# Adam optimizer to update weights
optimizer = optim.Adam(model.parameters())

print(model)

RNN(
  (rnn): RNN(5000, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


## 5. Train the RNN

In [23]:
epochs = 10

for epoch in range(epochs):
    model.train()   # set model to training mode

    for Xb, yb in train_loader:
        optimizer.zero_grad()          # clear gradients from previous step

        Xb = Xb.unsqueeze(1)           # add sequence dimension: (64, 5000) -> (64, 1, 5000)

        outputs = model(Xb)            # forward pass — get raw predictions (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze())   # convert to probability (0 to 1)

        loss = criterion(outputs, yb)  # compute loss (how wrong the prediction was)
        loss.backward()                # backpropagation — compute gradients
        optimizer.step()               # update model weights

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item():.4f}")

epoch = 1/10 and loss = 0.2421
epoch = 2/10 and loss = 0.2118
epoch = 3/10 and loss = 0.1914
epoch = 4/10 and loss = 0.3086
epoch = 5/10 and loss = 0.2784
epoch = 6/10 and loss = 0.3323
epoch = 7/10 and loss = 0.3001
epoch = 8/10 and loss = 0.2753
epoch = 9/10 and loss = 0.2549
epoch = 10/10 and loss = 0.1712


## 6. Evaluate the Model

In [24]:
model.eval()   # set model to evaluation mode (disables dropout etc)

with torch.no_grad():   # no need to compute gradients during evaluation
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        # if probability > 0.5 -> positive (1), else negative (0)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    accuracy = correct_vals / tot_vals * 100
    print(f"Test Accuracy = {accuracy:.2f}%")

Test Accuracy = 85.32%
